In [1]:
import os
import re 

import jsonlines
import pandas as pd
import polars as pl

from autogen_core.models import UserMessage
from autogen_agentchat.agents import AssistantAgent, UserProxyAgent
from autogen_agentchat.messages import TextMessage
from autogen_core import CancellationToken
from autogen_ext.models.ollama import OllamaChatCompletionClient

In [2]:
data_path       = f'../data'
# tables_path     = f'{data_path}/datasets/CAN/tables/tables_from10000_to15000'
# metadata_path   = f'{data_path}/datasets/CAN/metadata/metadata_from10000_to15000'
tables_path     = f'{data_path}/datasets/CAN/tables/tables_from0_to10000'
metadata_path   = f'{data_path}/datasets/CAN/metadata/metadata_from0_to10000'

candidates_path = f'{data_path}/outputs/candidate_joins.csv'

In [3]:
table_ids = list(sorted(os.listdir(tables_path), reverse=True))

In [4]:
with jsonlines.open(metadata_path) as fr:
    metadata = {rsc['id']: md for md in fr.iter() for rsc in md['resources'] if rsc['format'] == 'CSV'}

In [5]:
def get_package_id(rsc_id):
    rsc_id = re.sub(r'(_\d+)?.parquet$', '', table_ids[rsc_id])
    return metadata[rsc_id]['id']
    
def get_resource_metadata(rsc_id):
    rsc_id = re.sub(r'(_\d+)?.parquet$', '', table_ids[rsc_id])
    md = next(
        filter(
            lambda r: r['id'] == rsc_id, metadata[rsc_id]['resources']))
    return md['name'], metadata[rsc_id]['title'], metadata[rsc_id]['notes']

In [6]:
candidates = pl.read_csv(candidates_path)
candidates

r_tab_id,s_tab_id,r_col_id,s_col_id,r_col_name,s_col_name,size_r_col,size_s_col,r_pkg_id,s_pkg_id,size_intersection,size_union,jaccard,overlap
i64,i64,i64,i64,str,str,i64,i64,str,str,i64,i64,f64,f64
0,7632,0,0,"""ï»¿COLUMN NAME""","""ï»¿COLUMN NAME""",11,9,"""82cad281-ff7d-47b3-b2ce-9f7942…","""be54680b-ea62-46f3-aaa9-7644ed…",9,11,0.818,1.0
0,7218,0,0,"""ï»¿COLUMN NAME""","""ï»¿COLUMN NAME""",11,9,"""82cad281-ff7d-47b3-b2ce-9f7942…","""f281b150-0645-48e4-9c30-01f55f…",9,11,0.818,1.0
0,5094,0,0,"""ï»¿COLUMN NAME""","""ï»¿COLUMN NAME""",11,12,"""82cad281-ff7d-47b3-b2ce-9f7942…","""f281b150-0645-48e4-9c30-01f55f…",5,18,0.278,0.455
0,5536,0,0,"""ï»¿COLUMN NAME""","""ï»¿COLUMN NAME""",11,12,"""82cad281-ff7d-47b3-b2ce-9f7942…","""be54680b-ea62-46f3-aaa9-7644ed…",5,18,0.278,0.455
0,94,0,0,"""ï»¿COLUMN NAME""","""ï»¿COLUMN NAME""",11,13,"""82cad281-ff7d-47b3-b2ce-9f7942…","""823c63da-a4f8-4dce-a0fb-2de2dc…",4,20,0.2,0.364
…,…,…,…,…,…,…,…,…,…,…,…,…,…
8835,7577,1,3,"""Institution""","""Unnamed: 3""",25,27,"""6358b21a-ce9f-4b73-a4d0-7e30f7…","""e4d0558d-0269-4377-9a92-3bc0ab…",25,27,0.926,1.0
8835,5846,1,3,"""Institution""","""Unnamed: 3""",25,27,"""6358b21a-ce9f-4b73-a4d0-7e30f7…","""670516e0-6cb0-4d3b-ab2b-b82ed9…",25,27,0.926,1.0
8835,8246,1,0,"""Institution""","""ï»¿Institution""",25,56,"""6358b21a-ce9f-4b73-a4d0-7e30f7…","""81558d54-1f96-46c2-94fe-56d26f…",25,56,0.446,1.0


In [9]:
final_candidates = candidates
    # .filter(10 <= pl.col('size_r_col')) \
    # .filter(10 <= pl.col('size_s_col'))
    # .filter(pl.col('r_col_name') != pl.col('s_col_name')) \
    # .filter(10 <= pl.col('size_intersection')) \

final_candidates

r_tab_id,s_tab_id,r_col_id,s_col_id,r_col_name,s_col_name,size_r_col,size_s_col,r_pkg_id,s_pkg_id,size_intersection,size_union,jaccard,overlap
i64,i64,i64,i64,str,str,i64,i64,str,str,i64,i64,f64,f64
0,7632,0,0,"""ï»¿COLUMN NAME""","""ï»¿COLUMN NAME""",11,9,"""82cad281-ff7d-47b3-b2ce-9f7942…","""be54680b-ea62-46f3-aaa9-7644ed…",9,11,0.818,1.0
0,7218,0,0,"""ï»¿COLUMN NAME""","""ï»¿COLUMN NAME""",11,9,"""82cad281-ff7d-47b3-b2ce-9f7942…","""f281b150-0645-48e4-9c30-01f55f…",9,11,0.818,1.0
0,5094,0,0,"""ï»¿COLUMN NAME""","""ï»¿COLUMN NAME""",11,12,"""82cad281-ff7d-47b3-b2ce-9f7942…","""f281b150-0645-48e4-9c30-01f55f…",5,18,0.278,0.455
0,5536,0,0,"""ï»¿COLUMN NAME""","""ï»¿COLUMN NAME""",11,12,"""82cad281-ff7d-47b3-b2ce-9f7942…","""be54680b-ea62-46f3-aaa9-7644ed…",5,18,0.278,0.455
0,94,0,0,"""ï»¿COLUMN NAME""","""ï»¿COLUMN NAME""",11,13,"""82cad281-ff7d-47b3-b2ce-9f7942…","""823c63da-a4f8-4dce-a0fb-2de2dc…",4,20,0.2,0.364
…,…,…,…,…,…,…,…,…,…,…,…,…,…
8835,7577,1,3,"""Institution""","""Unnamed: 3""",25,27,"""6358b21a-ce9f-4b73-a4d0-7e30f7…","""e4d0558d-0269-4377-9a92-3bc0ab…",25,27,0.926,1.0
8835,5846,1,3,"""Institution""","""Unnamed: 3""",25,27,"""6358b21a-ce9f-4b73-a4d0-7e30f7…","""670516e0-6cb0-4d3b-ab2b-b82ed9…",25,27,0.926,1.0
8835,8246,1,0,"""Institution""","""ï»¿Institution""",25,56,"""6358b21a-ce9f-4b73-a4d0-7e30f7…","""81558d54-1f96-46c2-94fe-56d26f…",25,56,0.446,1.0


Get an example row

In [58]:
i = 10

row = final_candidates.row(i)
r_tab_id, s_tab_id, r_col_id, s_col_id, r_col_name, s_col_name = row[:6]
row

(10,
 4945,
 1,
 0,
 'GEO',
 'Number of temporary foreign worker (TFW) positions on requested Labour Market Impact Assessments (LMIAs) by province/territory',
 14,
 23,
 'a93bf95b-3f04-4a00-8457-a12c62d1279a',
 'e8745429-21e7-4a73-b3f5-90a779b78d1e',
 14,
 23,
 0.609,
 1.0)

In [59]:
r_rsc_name, r_pkg_name, r_pkg_note = get_resource_metadata(r_tab_id)
s_rsc_name, s_pkg_name, s_pkg_note = get_resource_metadata(s_tab_id)

import re
r_pkg_note = re.sub(r"(\n|\r|\t)", " ", r_pkg_note)
s_pkg_note = re.sub(r"(\n|\r|\t)", " ", s_pkg_note)

print(f'> {r_rsc_name=}')
print(f'> {s_rsc_name=}')
print(f'>> {r_col_id=}')
print(f'>> {s_col_id=}')
print(f'>>> {r_col_name=}')
print(f'>>> {s_col_name=}')
print(f'>>>> {r_pkg_note[:500]=}')
print(f'>>>> {s_pkg_note[:500]=}')

> r_rsc_name='Dataset'
> s_rsc_name='Table 12-Number of Temporary Foreign Worker (TFW) Positions on Requested Labour Market Impact Assessments (LMIAs) by Province/Territory between 2023Q1 and 2024Q3'
>> r_col_id=1
>> s_col_id=0
>>> r_col_name='GEO'
>>> s_col_name='Number of temporary foreign worker (TFW) positions on requested Labour Market Impact Assessments (LMIAs) by province/territory'
>>>> r_pkg_note[:500]='Average weekly earnings (including overtime) for all employees by enterprise size and North American Industry Classification System (NAICS), last 5 quarters.'
>>>> s_pkg_note[:500]='Overview:    Each quarter, the Temporary Foreign Worker Program (TFWP) publishes Labour Market Impact Assessment (LMIA) statistics on Open Government Data Portal, including quarterly and annual LMIA data related to, but not limited to, requested and approved TFW positions, employment location, employment occupations, sectors, TFWP stream and temporary foreign workers by country of origin.  The TFWP 

In [60]:
r_df = pl.read_parquet(f'{tables_path}/{table_ids[r_tab_id]}')
r_df

"ï»¿""REF_DATE""",GEO,DGUID,Enterprise size of employment,North American Industry Classification System (NAICS),UOM,UOM_ID,SCALAR_FACTOR,SCALAR_ID,VECTOR,COORDINATE,VALUE,STATUS,SYMBOL,TERMINATED,DECIMALS
str,str,str,str,str,str,i64,str,i64,str,str,f64,str,f64,f64,i64
"""2001-01""","""Canada""","""2021A000011124""","""All sizes""","""Industrial aggregate excluding…","""Current dollars""",75,"""units""",0,"""v4248809""","""1.1.2""",656.5,null,null,null,2
"""2001-01""","""Canada""","""2021A000011124""","""All sizes""","""Forestry, logging and support …","""Current dollars""",75,"""units""",0,"""v4248810""","""1.1.4""",822.5,null,null,null,2
"""2001-01""","""Canada""","""2021A000011124""","""All sizes""","""Mining, quarrying, and oil and…","""Current dollars""",75,"""units""",0,"""v4248811""","""1.1.10""",1158.55,null,null,null,2
"""2001-01""","""Canada""","""2021A000011124""","""All sizes""","""Utilities [22]""","""Current dollars""",75,"""units""",0,"""v4248812""","""1.1.17""",1140.83,null,null,null,2
"""2001-01""","""Canada""","""2021A000011124""","""All sizes""","""Construction [23]""","""Current dollars""",75,"""units""",0,"""v4248813""","""1.1.21""",791.88,null,null,null,2
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""2024-07""","""Nunavut""","""2021A000262""","""300 and more employees""","""Health care and social assista…","""Current dollars""",75,"""units""",0,"""v4258464""","""14.11.331""",null,"""x""",null,null,2
"""2024-07""","""Nunavut""","""2021A000262""","""300 and more employees""","""Arts, entertainment and recrea…","""Current dollars""",75,"""units""",0,"""v4258465""","""14.11.354""",null,"""x""",null,null,2
"""2024-07""","""Nunavut""","""2021A000262""","""300 and more employees""","""Accommodation and food service…","""Current dollars""",75,"""units""",0,"""v4258466""","""14.11.367""",null,"""F""",null,null,2


In [61]:
s_df = pl.read_parquet(f'{tables_path}/{table_ids[s_tab_id]}')
s_df

Number of temporary foreign worker (TFW) positions on requested Labour Market Impact Assessments (LMIAs) by province/territory,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7
str,str,str,str,str,str,str,str
null,null,null,null,null,null,null,null
null,null,null,null,null,null,null,null
null,"""2023""",null,null,null,"""2024""",null,null
"""Province/Territory""","""Jan-Mar""","""Apr-Jun""","""Jul-Sep""","""Oct-Dec""","""Jan-Mar""","""Apr-Jun""","""Jul-Sep"""
"""Newfoundland and Labrador""","""599""","""301""","""183""","""869""","""508""","""701""","""750"""
…,…,…,…,…,…,…,…
"""2. As of the publication of Q4…",null,null,null,null,null,null,null
"""3. The sum of the number of TF…",null,null,null,null,null,null,null
"""4. Employers may only submit o…",null,null,null,null,null,null,null


In [62]:
from orqa.utils import sanitize_string

common_cells = list(set(map(sanitize_string, r_df.to_series(r_col_id))) & set(map(sanitize_string, s_df.to_series(s_col_id))))
common_cells

['newfoundland_and_labrador',
 'prince_edward_island',
 'yukon',
 'nunavut',
 'canada',
 'new_brunswick',
 'manitoba',
 'alberta',
 'nova_scotia',
 'ontario',
 'saskatchewan',
 'northwest_territories',
 'british_columbia',
 'quebec']

# Join Evaluation with Agents (Autogen v0.4)

### Agent/wTool Setup

In [ ]:
import asyncio
import json
from dataclasses import dataclass
from typing import List

from autogen_core import (
    AgentId,
    FunctionCall,
    MessageContext,
    RoutedAgent,
    SingleThreadedAgentRuntime,
    message_handler,
)
from autogen_core.models import (
    ChatCompletionClient,
    LLMMessage,
    AssistantMessage,
    SystemMessage,
    UserMessage,
    FunctionExecutionResult, 
    FunctionExecutionResultMessage,
    ModelFamily
)
from autogen_core.tools import FunctionTool, Tool
from autogen_ext.models.ollama import OllamaChatCompletionClient
from autogen_ext.models.openai import OpenAIChatCompletionClient

In [ ]:
@dataclass
class Message:
    content: str


class ToolUseAgent(RoutedAgent):
    def __init__(self, model_client: OpenAIChatCompletionClient|OllamaChatCompletionClient, system_message: str, tool_schema: List[Tool]) -> None:
        super().__init__("JOIN Assistant with Tool")
        self._system_messages: List[LLMMessage] = [
            SystemMessage(content=system_message)
        ]
        self._model_client = model_client
        self._tools = tool_schema

    @message_handler
    async def handle_user_message(self, message: Message, ctx: MessageContext) -> Message:
        # Create a session of messages.
        session: List[LLMMessage] = self._system_messages + [UserMessage(content=message.content, source="user")]

        # Run the chat completion with the tools.
        create_result = await self._model_client.create(
            messages=session,
            tools=self._tools,
            cancellation_token=ctx.cancellation_token,
        )

        # If there are no tool calls, return the result.
        if isinstance(create_result.content, str):
            return Message(content=create_result.content)
        assert isinstance(create_result.content, list) and all(
            isinstance(call, FunctionCall) for call in create_result.content
        )

        # Add the first model create result to the session.
        session.append(AssistantMessage(content=create_result.content, source="assistant"))

        # Execute the tool calls.
        results = await asyncio.gather(
            *[self._execute_tool_call(call, ctx.cancellation_token) for call in create_result.content]
        )

        # Add the function execution results to the session.
        session.append(FunctionExecutionResultMessage(content=results))

        # Run the chat completion again to reflect on the history and function execution results.
        create_result = await self._model_client.create(
            messages=session,
            cancellation_token=ctx.cancellation_token,
        )
        assert isinstance(create_result.content, str)

        # Return the result as a message.
        return Message(content=create_result.content)

    async def _execute_tool_call(
        self, call: FunctionCall, cancellation_token: CancellationToken
    ) -> FunctionExecutionResult:
        # Find the tool by name.
        tool = next((tool for tool in self._tools if tool.name == call.name), None)
        assert tool is not None
        # Run the tool and capture the result.
        try:
            arguments = json.loads(call.arguments)
            result = await tool.run_json(arguments, cancellation_token)
            return FunctionExecutionResult(
                call_id=call.id, content=tool.return_value_as_string(result), is_error=False, name=tool.name
            )
        except Exception as e:
            return FunctionExecutionResult(call_id=call.id, content=str(e), is_error=True, name=tool.name)

In [31]:
from typing import List
from typing_extensions import Annotated
from autogen_core.tools import Tool, FunctionTool

results = []

# in this case prob is not necessary to equip the assistant agent with this tool
# we can simply call it with agent.create()
async def save_join_score(score: Annotated[int, 
                          """A score of the JOIN between 0 and 3 where:
                              0 is CASUAL;
                              1 is SUPERFICIAL;
                              2 is ENGAGED;
                              3 is MEANINGFUL;"""
                          ],
                         explanation: Annotated[str, "A clear, short and concise explanation of the given score"]):
    # print("Tool has been called!")
    results.append((score, explanation))
    return score, explanation

join_score_tool = FunctionTool(save_join_score, description="Save the JOIN evaluation and explanation")

In [85]:
# Create the client
model_client = OllamaChatCompletionClient(
# model_client = OpenAIChatCompletionClient(
    # model="ollama/llama3.3",
    # model="deepseek-r1:70b",
    model="qwq:32b-fp16",
    temperature=0,
    api_key="NotRequiredSinceWeAreLocal",
    base_url="http://localhost:4000",
    model_info={
        "json_output": False,
        "vision": False,
        "function_calling": True,
        # "family": ModelFamily.R1,
        "family": ModelFamily.UNKNOWN,
        "keep_alive": -1
    },
    keep_alive=-1,
    num_ctx=8192
    # options={
    #     "num_ctx": 4096,
    #     "temperature": 0,
    #     "keep_alive": "1h"
    # }
)

system_message = """
    You are a smart AI assistant.
    Do not write code. Do not respond. Only use the provided tool "save_join_score".
"""

# Create a runtime.
runtime = SingleThreadedAgentRuntime()

# Create the tools.
tools: List[Tool] = [join_score_tool]

# Register the agents.
await ToolUseAgent.register(
    runtime,
    "tool_use_agent",
    lambda: ToolUseAgent(
        model_client,
        system_message,
        tools,
    ),
)

AgentType(type='tool_use_agent')

### Agent Interaction

In [76]:
import warnings
warnings.filterwarnings('error')

In [86]:
try:
    runtime.start()
except RuntimeError:
    await runtime.stop()
    runtime.start()

tool_use_agent = AgentId("tool_use_agent", "default")

response = await runtime.send_message(
    Message(f"""
        Define a score for: 
        {r_col_name=}, {s_col_name=}, 
        r_table_name={r_rsc_name}, s_table_name={s_rsc_name}, 
        
        r_table_description={r_pkg_note[:500]}, s_table_description={s_pkg_note[:500]}
        
        common_cells: {common_cells[:30]}
        
        r_table_sample:
        {r_df.sample(5)}, 
        
        s_table_sample:
        {s_df.sample(5)}"""),
    tool_use_agent)
await runtime.stop()

In [87]:
from pprint import pprint
print(response.content)
pprint(results)

<think>
Okay, let's tackle this problem step by step. I need to figure out the correct answer based on the given information. The user has provided details about two tables, r_table and s_table, along with some samples and asks for a score between 0 and 1 indicating their relatedness. 

First, let me understand what exactly is being asked here. The task is to determine how related or similar the two tables are, probably based on their structure, content, or the columns they share. The final output should be a numerical score between 0 (not related) and 1 (very related).

Looking at the provided data:

**r_table:**
- Columns include "Number of temporary foreign wo…" and several "Unnamed" columns with numbers. The sample data shows entries like "Alberta" followed by numbers, which might represent counts for different quarters or years. There are also some rows with text explanations, like "2. As of the publication of Q4…" which might be footnotes or additional information. The rows are s

# Basic Loop 

In [10]:
candidates.columns

['r_tab_id',
 's_tab_id',
 'r_col_id',
 's_col_id',
 'r_col_name',
 's_col_name',
 'size_r_col',
 'size_s_col',
 'r_pkg_id',
 's_pkg_id',
 'size_intersection',
 'size_union',
 'jaccard',
 'overlap']

In [ ]:
# Process one row at a time, getting if this is acceptable or note

evaluations = []

for row in candidates.rows():
    r_tab_id, s_tab_id, r_col_id, s_col_id, r_col_name, s_col_name = row[:6]
    